In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [6]:
df = pd.read_parquet("5G_DataSet.parquet", engine="pyarrow")

In [7]:
y = df[['Longitude', 'Latitude']]
X = df.drop(columns=['Longitude', 'Latitude', 'Time'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
deg_to_meter = 111320  # 1 derece ≈ 111320 metre

models = {
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, objective='reg:squarederror'),
    "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1),
    "LGBM": LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, random_seed=42, verbose=0)
}

top_10_features_per_model = {}

for name, base_model in models.items():
    print(f"\n\n===== {name} =====")
    
    rmse_long_list = []
    mae_long_list = []
    rmse_lat_list = []
    mae_lat_list = []

    for fold, (train_index, val_index) in enumerate(kf.split(X_train)):
        X_tr, X_val = X_train.iloc[train_index], X_train.iloc[val_index]
        y_tr, y_val = y_train.iloc[train_index], y_train.iloc[val_index]

        multi_model = MultiOutputRegressor(base_model)
        multi_model.fit(X_tr, y_tr)
        y_pred_val = multi_model.predict(X_val)

        rmse_long = np.sqrt(mean_squared_error(y_val['Longitude'], y_pred_val[:, 0]))
        mae_long = mean_absolute_error(y_val['Longitude'], y_pred_val[:, 0])
        rmse_lat = np.sqrt(mean_squared_error(y_val['Latitude'], y_pred_val[:, 1]))
        mae_lat = mean_absolute_error(y_val['Latitude'], y_pred_val[:, 1])

        rmse_long_list.append(rmse_long)
        mae_long_list.append(mae_long)
        rmse_lat_list.append(rmse_lat)
        mae_lat_list.append(mae_lat)

        print(f"Fold {fold+1}:")
        print(f"  Longitude RMSE: {rmse_long:.6f} deg ≈ {rmse_long * deg_to_meter:.2f} m")
        print(f"  Longitude MAE : {mae_long:.6f} deg ≈ {mae_long * deg_to_meter:.2f} m")
        print(f"  Latitude  RMSE: {rmse_lat:.6f} deg ≈ {rmse_lat * deg_to_meter:.2f} m")
        print(f"  Latitude  MAE : {mae_lat:.6f} deg ≈ {mae_lat * deg_to_meter:.2f} m\n")

    print(">>> Cross Validation Ortalamaları:")
    print(f"Longitude CV RMSE ort.: {np.mean(rmse_long_list):.6f} deg ≈ {np.mean(rmse_long_list) * deg_to_meter:.2f} m")
    print(f"Longitude CV MAE  ort.: {np.mean(mae_long_list):.6f} deg ≈ {np.mean(mae_long_list) * deg_to_meter:.2f} m")
    print(f"Latitude  CV RMSE ort.: {np.mean(rmse_lat_list):.6f} deg ≈ {np.mean(rmse_lat_list) * deg_to_meter:.2f} m")
    print(f"Latitude  CV MAE  ort.: {np.mean(mae_lat_list):.6f} deg ≈ {np.mean(mae_lat_list) * deg_to_meter:.2f} m")

   
    final_model = MultiOutputRegressor(base_model)
    final_model.fit(X_train, y_train)
    y_pred_test = final_model.predict(X_test)

    rmse_long_test = np.sqrt(mean_squared_error(y_test['Longitude'], y_pred_test[:, 0]))
    mae_long_test = mean_absolute_error(y_test['Longitude'], y_pred_test[:, 0])
    rmse_lat_test = np.sqrt(mean_squared_error(y_test['Latitude'], y_pred_test[:, 1]))
    mae_lat_test = mean_absolute_error(y_test['Latitude'], y_pred_test[:, 1])

    print("\n>>> Test Set Performansı:")
    print(f"Longitude Test RMSE: {rmse_long_test:.6f} deg ≈ {rmse_long_test * deg_to_meter:.2f} m")
    print(f"Longitude Test MAE : {mae_long_test:.6f} deg ≈ {mae_long_test * deg_to_meter:.2f} m")
    print(f"Latitude  Test RMSE: {rmse_lat_test:.6f} deg ≈ {rmse_lat_test * deg_to_meter:.2f} m")
    print(f"Latitude  Test MAE : {mae_lat_test:.6f} deg ≈ {mae_lat_test * deg_to_meter:.2f} m")

    estimator = multi_model.estimators_[0]

   
    if name == "CatBoost":
        importances = estimator.get_feature_importance()
    else:
        importances = estimator.feature_importances_

   
    fi_df = pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False).reset_index(drop=True)

    top_10 = fi_df.head(10)
    top_10_features_per_model[name] = top_10


for model_name, df in top_10_features_per_model.items():
    print(f"\n===== {model_name} - İlk 10 Önemli Özellik (Longitude) =====")
    display(df.reset_index(drop=True))




===== XGBoost =====
Fold 1:
  Longitude RMSE: 0.000158 deg ≈ 17.62 m
  Longitude MAE : 0.000105 deg ≈ 11.72 m
  Latitude  RMSE: 0.000101 deg ≈ 11.22 m
  Latitude  MAE : 0.000073 deg ≈ 8.12 m

Fold 2:
  Longitude RMSE: 0.000147 deg ≈ 16.42 m
  Longitude MAE : 0.000102 deg ≈ 11.32 m
  Latitude  RMSE: 0.000103 deg ≈ 11.52 m
  Latitude  MAE : 0.000075 deg ≈ 8.34 m

Fold 3:
  Longitude RMSE: 0.000149 deg ≈ 16.53 m
  Longitude MAE : 0.000104 deg ≈ 11.63 m
  Latitude  RMSE: 0.000105 deg ≈ 11.69 m
  Latitude  MAE : 0.000075 deg ≈ 8.36 m

Fold 4:
  Longitude RMSE: 0.000149 deg ≈ 16.61 m
  Longitude MAE : 0.000104 deg ≈ 11.59 m
  Latitude  RMSE: 0.000103 deg ≈ 11.51 m
  Latitude  MAE : 0.000076 deg ≈ 8.42 m

Fold 5:
  Longitude RMSE: 0.000140 deg ≈ 15.53 m
  Longitude MAE : 0.000101 deg ≈ 11.23 m
  Latitude  RMSE: 0.000111 deg ≈ 12.33 m
  Latitude  MAE : 0.000078 deg ≈ 8.63 m

>>> Cross Validation Ortalamaları:
Longitude CV RMSE ort.: 0.000149 deg ≈ 16.54 m
Longitude CV MAE  ort.: 0.000103 deg

,Feature,Importance
0,UL_NR_UE_Nbr_RSRP_1,0.308068
1,UL_NR_UE_Nbr_RSRP_2,0.172981
2,SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_5,0.115870
3,UL_NR_UE_PCI_0_38.0,0.054417
4,DL_NR_UE_PCI_0_76.0,0.030324
5,DL_NR_UE_PCI_0_48.0,0.026569
6,UL_NR_UE_Nbr_PCI_2_76.0,0.022328
7,UL_NR_UE_Nbr_PCI_0_48.0,0.020607
8,UL_NR_UE_PCI_0_68.0,0.018516
9,DL_NR_UE_PCI_0_68.0,0.018154



===== RandomForest - İlk 10 Önemli Özellik (Longitude) =====


,Feature,Importance
0,UL_NR_UE_Nbr_RSRP_1,0.571025
1,UL_NR_UE_Nbr_RSRP_2,0.183027
2,SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_5,0.083501
3,SC_NR_Scan_PCI_SortedBy_RSRP_2,0.036581
4,UL_NR_UE_PCI_0_38.0,0.020705
5,UL_NR_UE_PCI_0_68.0,0.012428
6,SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_1,0.010097
7,DL_NR_UE_PCI_0_48.0,0.009042
8,SC_NR_Scan_PCI_SortedBy_RSRP_1,0.008986
9,UL_NR_UE_Nbr_PCI_2_76.0,0.007932



===== LGBM - İlk 10 Önemli Özellik (Longitude) =====


,Feature,Importance
0,SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_1,104
1,SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_2,102
2,SC_NR_Scan_PCI_SortedBy_RSRP_2,87
3,UL_NR_UE_Nbr_RSRP_1,87
4,DL_NR_UE_RB_Num_DL_0,79
5,SC_NR_Scan_PCI_SortedBy_RSRP_1,77
6,SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_3,68
7,DL_NR_UE_RSRP_0,64
8,SC_NR_Scan_SSB_RSRQ_SortedBy_RSRP_2,61
9,SC_NR_Scan_SSB_RSRQ_SortedBy_RSRP_1,57



===== CatBoost - İlk 10 Önemli Özellik (Longitude) =====


,Feature,Importance
0,UL_NR_UE_Nbr_RSRP_1,35.469430
1,UL_NR_UE_Nbr_RSRP_2,11.077175
2,SC_NR_Scan_PCI_SortedBy_RSRP_2,6.179259
3,UL_NR_UE_Power_Tx_PRACH_0,5.502251
4,SC_NR_Scan_SSB_RSRP_SortedBy_RSRP_5,5.206166
5,UL_NR_UE_Nbr_RSRQ_2,4.644408
6,SC_NR_Scan_PCI_SortedBy_RSRP_1,3.460571
7,DL_NR_UE_Power_Tx_PRACH_0,2.369412
8,UL_NR_UE_Nbr_RSRP_3,2.157268
9,DL_NR_UE_Nbr_RSRP_2,1.996102
